In [ ]:
#| default_exp search

# search

> private web search through a local SearXNG instance

The search module routes queries through SearXNG, a self-hosted meta-search engine running in Docker. fossick starts the container automatically on the first call. Results are cached in Redis, so repeated queries return immediately.

In [ ]:
#| export
import os, time, atexit, shutil
import httpx
from fastcore.all import AttrDictDefault, L, Path
from fossick.core import save_path, http_get

## SearXNG

In [ ]:
#| export
_SEARXNG_URL   = 'http://localhost:8080'
_SEARXNG_NAME  = 'fossick-searxng'
_SEARXNG_NET   = 'fossick-net'
_SEARXNG_DIR   = save_path('searxng')
_COMPOSE_PATH  = str(_SEARXNG_DIR / 'docker-compose.yml')

def _wait_for_searxng(url:str, timeout:int=30, interval:float=0.5):
    "Poll SearXNG until HTTP 200 or raise RuntimeError on timeout."
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            if httpx.get(url, timeout=2).status_code == 200: return
        except Exception: pass
        time.sleep(interval)
    raise RuntimeError(f'SearXNG did not start within {timeout}s at {url}. Is Docker running?')

In [ ]:
#| export
def searxng_start(settings:Path=None) -> str:
    "Start SearXNG + Redis via Compose. `settings` overrides the bundled searxng_settings.yml. Idempotent."
    from dockeasy import Compose
    url = os.environ.get('SEARXNG_URL')
    if url: return url
    try:
        if http_get(_SEARXNG_URL).status_code == 200: return _SEARXNG_URL
    except Exception: pass
    _SEARXNG_DIR.mkdir(parents=True, exist_ok=True)
    f = __file__ if '__file__' in globals() else '.'
    src = Path(settings) if settings else Path(f).parent / 'searxng_settings.yml'
    shutil.copy(src, _SEARXNG_DIR / 'settings.yml')
    (Compose()
        .network(_SEARXNG_NET)
        .svc('redis',
             image='redis:7-alpine',
             command='redis-server --maxmemory 256mb --maxmemory-policy allkeys-lru --save ""',
             networks=[_SEARXNG_NET],
             restart='unless-stopped')
        .svc('searxng',
             image='searxng/searxng:latest',
             ports={'8080': '8080'},
             volumes={str(_SEARXNG_DIR): '/etc/searxng'},
             env={'SEARXNG_SECRET': 'fossick'},
             container_name=_SEARXNG_NAME,
             depends_on=['redis'],
             networks=[_SEARXNG_NET])
        .up(detach=True, path=_COMPOSE_PATH))
    _wait_for_searxng(_SEARXNG_URL)
    return _SEARXNG_URL

def searxng_stop():
    "Stop SearXNG + Redis if we started them. No-op otherwise."
    from dockeasy import Compose
    try: Compose().down(path=_COMPOSE_PATH)
    except Exception: pass

atexit.register(searxng_stop)

<function __main__.searxng_stop()>

## Result type and query routing

`_intent()` scores the query against keyword lists for each category. At least two signals must match before routing away from the default `general` category, which keeps single generic words like "python" from sending a query to GitHub and Stack Overflow instead of Google.

In [ ]:
#| export
class Result(AttrDictDefault):
    "One search result. Fields: title, url, snippet, score, engines, provider, ts."

# SearXNG category by query intent
_category_map = dict(academic='science', code='it', recent='news', default='general')
_intents = dict(
    academic=['research', 'paper', 'study', 'arxiv', 'doi', 'journal',
              'citation', 'how does', 'why is', 'explain', 'mechanism'],
    code=['github', 'stackoverflow', 'npm', 'pypi', 'repo'],
    recent=['today', 'yesterday', 'this week', 'latest', 'breaking',
              'current', 'just announced', '2025', '2026'])

def _intent(q:str) -> tuple:
    "Return (intent, searxng_category) for query q. Requires >=2 signals to route away from general."
    ql = q.lower()
    scores = {k: sum(1 for p in ps if p in ql) for k, ps in _intents.items()}
    intent = max(scores, key=scores.get) if max(scores.values()) >= 2 else 'default'
    return intent, _category_map.get(intent, 'general')

## SearXNG search

In [ ]:
#| export
def _searxng(q:str, n:int=10, category:str=None, engines:str=None) -> L:
    "Search via SearXNG JSON API. Returns L of Result."
    try:
        url = searxng_start()
        params = {'q': q, 'format': 'json', 'pageno': 1}
        if category: params['categories'] = category
        if engines:  params['engines']    = engines
        r = http_get(f'{url}/search', params=params)
        if r.status_code != 200: return L()
        mk_res = lambda x: Result(title=x.get('title', ''),
            url=x.get('url', ''),snippet=x.get('content', ''),
            score=x.get('score', 0.0), engines=x.get('engines', []),
            provider='searxng', ts=time.time())
        return L(r.json().get('results', [])[:n]).map(mk_res)
    except Exception as e:
        print(f'searxng error: {e}')
        return L()

In [ ]:
#| export
def search(q:str,
           n:int=10,           # max results
           category:str=None,  # override SearXNG category (general/science/it/news)
           engines:str=None,   # comma-sep engine names passed to SearXNG
          ) -> L:
    "Search the web. Returns L of Result sorted by relevance."
    if not q or not q.strip(): return L()
    intent, cat = _intent(q)
    cat = category or cat
    return _searxng(q, n, category=cat, engines=engines)

## Tests

In [ ]:
# Verify intent routing
assert _intent('arxiv transformer paper')[0] == 'academic'
assert _intent('latest news today')[0] == 'recent'
assert _intent('what is the capital of France')[0] == 'default'
# Requires >=2 code signals — single match falls through to general
assert _intent('github python error')[0] == 'default'        # only 1 signal (github)
assert _intent('github npm package install')[0] == 'code'    # 2 signals (github + npm)
# Generic web-dev terms alone must not trigger code routing
assert _intent('fasthtml python web framework')[0] == 'default'

In [ ]:
searxng_start()

'http://localhost:8080'

In [ ]:
#| eval: false
# Live test — requires Docker
url = searxng_start()
print('SearXNG at', url)
results = search('fasthtml python web framework', n=5)
for r in results: print(r.title, r.url)

SearXNG at http://localhost:8080
FastHTML - Modern web applications in pure Python https://fastht.ml/
GitHub - AnswerDotAI/fasthtml: The fastest way to create an HTML app https://github.com/answerdotai/fasthtml
FastHTML: a hypermedia-first python framework w/htmx available ootb https://www.reddit.com/r/htmx/comments/1efizmn/fasthtml_a_hypermediafirst_python_framework_whtmx/
FastHTML Tutorial: Build Modern Web Applications with Pure Python https://www.youtube.com/watch?v=AxA8YH_UyBo
First encounter with FastHTML: Building a FastHTML assistant https://medium.com/@mrsirsh/first-encounter-with-fasthtml-building-a-fasthtml-assistant-fe896d3a3e60


In [ ]:
#| eval: false
search('what is the capital of France')[0]

```python
{ 'default_': None,
  'engines': ['google', 'startpage'],
  'provider': 'searxng',
  'score': 8.0,
  'snippet': 'Paris is the capital and largest city of France, with an '
             'estimated city population of 2.04 million in an area of 105.4 km '
             '2 (40.7 sq mi), and a metropolitan ...',
  'title': 'Paris - Wikipedia',
  'ts': 1780439809.467083,
  'url': 'https://en.wikipedia.org/wiki/Paris'}
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()